# 05 · Report generation (computed facts, optional LLM narration)

Every number in the report comes from the analysis code. An LLM is optional and is used only to explain the computed facts in plain English.

How it works:
1. `build_facts()` collects computed results into a small dictionary
2. `render_report()` turns the facts into a Markdown report deterministically
3. If `OPENAI_API_KEY` is set, `InsightGenerator` asks the model to explain the facts, with instructions to use only those numbers, and `ungrounded_numbers()` flags any number in the model's text that is not in the facts

**Earlier version:** this notebook printed hand-written "mock" LLM responses (including an investor letter with figures such as YTD returns and credit spreads) that were never computed, and saved them as the analysis report. That report has been removed.

In [1]:
import sys, json
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import *

pd.set_option('display.width', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.3})
COLORS = {'Calm': '#2a9d8f', 'Elevated': '#e9c46a', 'Crisis': '#e63946'}
READ = dict(index_col=0, parse_dates=True, float_precision='round_trip')

In [2]:
returns = pd.read_csv('../data/processed/market_returns.csv', **READ)
spx = returns['^GSPC']
assets = ['^GSPC', '^IXIC', 'GC=F', 'CL=F', 'DX-Y.NYB']
summary = json.load(open('../results/summary_metrics.json'))
robust = pd.read_csv('../results/robustness.csv')

# Regimes (descriptive, full sample)
detector = RegimeDetector(spx, random_state=42)
detector.fit_gmm(verbose=False)
characteristics = detector.get_regime_characteristics()
transition = detector.analyze_regime_transitions()['transition_matrix']

# Walk-forward headline run, same settings as notebook 04
st = summary['settings']
wf = walk_forward_regime_backtest(spx, train_window=st['train_window_days'], feature_window=st['feature_window_days'],
                                  lag=st['signal_lag_days'], cost_per_unit_turnover=st['cost_bps'] / 10000,
                                  random_state=st['random_state'])
weight = wf['positions'].mean()
walk_forward = {'Strategy': performance_metrics(wf['strategy_returns']),
                'Benchmark': performance_metrics(wf['benchmark_returns']),
                'Constant mix': performance_metrics(constant_mix_returns(wf['benchmark_returns'], weight)),
                'equity_weight': weight}
assert abs(walk_forward['Strategy']['Annual Return'] - summary['walk_forward']['regime_strategy']['cagr']) < 1e-4

# Cross-asset results
analyzer = CausalityAnalyzer(returns, assets)
analyzer.granger_causality_matrix(lag=1)
significant = analyzer.get_significant_causality(threshold=0.05, adjust='bonferroni')
corr = returns[assets].corr()
correlations = {'S&P 500 vs NASDAQ': corr.loc['^GSPC', '^IXIC'], 'S&P 500 vs oil': corr.loc['^GSPC', 'CL=F'],
                'S&P 500 vs gold': corr.loc['^GSPC', 'GC=F'], 'S&P 500 vs US dollar': corr.loc['^GSPC', 'DX-Y.NYB'],
                'Gold vs US dollar': corr.loc['GC=F', 'DX-Y.NYB']}

facts = build_facts(characteristics, transition, walk_forward, robustness=robust, significant_granger=significant,
                    correlations=correlations, period=f"{returns.index[0].date()} to {returns.index[-1].date()}",
                    settings={'refit': 'monthly', 'train_window_days': st['train_window_days'], 'signal_lag_days': st['signal_lag_days'],
                              'cost_bps': st['cost_bps'], 'granger_tests': len(assets) * (len(assets) - 1)})
print(json.dumps(facts, indent=2))

{
  "period": "2010-01-05 to 2024-12-30",
  "settings": {
    "refit": "monthly",
    "train_window_days": 504,
    "signal_lag_days": 1,
    "cost_bps": 5.0,
    "granger_tests": 20
  },
  "regimes": {
    "Calm": {
      "share_of_days_pct": 63.5,
      "daily_volatility_next_day_pct": 0.71,
      "mean_daily_return_next_day_pct": 0.042
    },
    "Elevated": {
      "share_of_days_pct": 34.9,
      "daily_volatility_next_day_pct": 1.34,
      "mean_daily_return_next_day_pct": 0.059
    },
    "Crisis": {
      "share_of_days_pct": 1.6,
      "daily_volatility_next_day_pct": 3.81,
      "mean_daily_return_next_day_pct": 0.238
    }
  },
  "probability_regime_continues_next_day_pct": {
    "Calm": 97.3,
    "Elevated": 94.9,
    "Crisis": 96.7
  },
  "walk_forward_strategy": {
    "cagr_pct": 8.9,
    "annual_volatility_pct": 9.5,
    "sharpe": 0.94,
    "max_drawdown_pct": -15.7,
    "start": "2012-02-02",
    "end": "2024-12-30"
  },
  "walk_forward_buy_and_hold": {
    "cagr_pct": 

In [3]:
report = InsightGenerator().generate_full_report(facts)   # uses the LLM only if OPENAI_API_KEY is set
with open('../results/analysis_report.md', 'w') as f:
    f.write(report['markdown'])
print(report['markdown'])
if report['llm_summary'] is None:
    print("(No OPENAI_API_KEY set: report contains computed facts only.)")

# Regime Risk Analysis: Summary Report

Data: S&P 500 and four cross-asset series, 2010-01-05 to 2024-12-30.

## Regimes (volatility states of the S&P 500)

| Regime | Share of days | Next-day daily volatility | Next-day mean return | P(still in regime tomorrow) |
|---|---|---|---|---|
| Calm | 63.5% | 0.71% | 0.042% | 97.3% |
| Elevated | 34.9% | 1.34% | 0.059% | 94.9% |
| Crisis | 1.6% | 3.81% | 0.238% | 96.7% |

Next-day volatility rises from regime to regime, so the regimes carry real information about near-term risk. Next-day mean returns are small and noisy by comparison, so the regimes say little about direction.

## Out-of-sample (walk-forward) backtest

Model refit on past data only (monthly, 504-day training window), signals applied after 1 day(s), 5 bps trading cost per unit of turnover. Period: 2012-02-02 to 2024-12-30.

| | Regime strategy | Fixed 66% equity mix | Buy & hold |
|---|---|---|---|
| CAGR | 8.9% | 8.4% | 12.3% |
| Annual volatility | 9.5% | 11.1% | 16.7% |
| S

## Optional: plain-English narration

With a key in `.env` (`OPENAI_API_KEY=...`), the cell below asks the model to explain the facts for a non-technical audience and lists any numbers it used that are not in the facts.

In [ ]:
from dotenv import load_dotenv
load_dotenv('../.env')
llm = LLMInsightGenerator(model='gpt-4o-mini')
if llm.is_configured():
    out = llm.narrate(facts, audience='a non-technical risk committee')
    print(out['text'])
    print('\nNumbers not found in the facts (review these):', out['ungrounded_numbers'] or 'none')
else:
    print('Skipped: OPENAI_API_KEY is not set.')